In [ ]:
import os
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import anndata as ad
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri

pandas2ri.activate()
%load_ext rpy2.ipython

In [ ]:
# Locate repository root and import path configuration
import os
import sys
from pathlib import Path

def _find_repo():
    env = os.environ.get("ATLAS_PIPELINE_ROOT", "").strip()
    if env:
        p = Path(env).expanduser().resolve()
        if (p / "pipeline_paths.py").is_file():
            return p
    cwd = Path.cwd().resolve()
    for d in (cwd, cwd.parent):
        if (d / "pipeline_paths.py").is_file():
            return d
    raise RuntimeError(
        "Set ATLAS_PIPELINE_ROOT to the repository root (folder containing pipeline_paths.py), "
        "or start Jupyter from that directory."
    )

REPO = _find_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
import pipeline_paths as pp


In [ ]:
# Set input and output directories
INPUT_DIR = str(pp.PREPROCESSED_H5AD)
OUTPUT_DIR = str(pp.PROCESSED_H5AD)

In [ ]:
# Install scrublet if not already installed
try:
    import scrublet as scr
    print("✓ scrublet is already installed")
except ImportError:
    print("Installing scrublet...")
    import subprocess
    subprocess.check_call(["pip", "install", "scrublet"])
    import scrublet as scr
    print("✓ scrublet installed")

In [ ]:
# Diagnostic: Check gene name formats in all h5ad files
import glob

def check_gene_names_in_files(input_dir):
    """
    Check gene name formats (Ensembl IDs vs gene symbols) in all h5ad files.
    """
    h5ad_files = glob.glob(os.path.join(input_dir, "*.h5ad"))
    
    if not h5ad_files:
        print(f"No .h5ad files found in {input_dir}")
        return
    
    print(f"Found {len(h5ad_files)} .h5ad file(s)\n")
    print("=" * 80)
    
    for h5ad_file in h5ad_files:
        file_name = os.path.basename(h5ad_file)
        print(f"\n📁 File: {file_name}")
        print("-" * 80)
        
        try:
            # Read just the var (gene) information (faster than full read)
            adata = sc.read_h5ad(h5ad_file, backed='r')  # Read-only mode
            
            print(f"  Total genes: {adata.n_vars}")
            print(f"  Total cells: {adata.n_obs}")
            
            # Check gene name format
            gene_names = adata.var_names
            
            # Check for Ensembl IDs
            ensembl_patterns = [
                gene_names.str.startswith('ENS'),
                gene_names.str.startswith('ens'),
                gene_names.str.match(r'^ENS[A-Z]*G\d+'),  # ENSG00000...
                gene_names.str.match(r'^ENS[A-Z]*T\d+'),  # ENST00000...
            ]
            is_ensembl = sum(ensembl_patterns) > 0
            
            # Check for common gene symbol patterns
            symbol_patterns = [
                gene_names.str.match(r'^[A-Z][A-Z0-9]+$'),  # All caps alphanumeric
                gene_names.str.match(r'^[A-Z][a-z]+$'),     # Capital + lowercase
                gene_names.str.startswith('MT-'),           # Mitochondrial
                gene_names.str.startswith('RPS'),            # Ribosomal
                gene_names.str.startswith('RPL'),
            ]
            is_symbol = sum(symbol_patterns) > 0
            
            # Sample gene names
            print(f"\n  Sample gene names (first 20):")
            for i, name in enumerate(gene_names[:20]):
                print(f"    {i+1:2d}. {name}")
            
            # Check for Ensembl IDs
            ensembl_count = gene_names.str.startswith('ENS').sum()
            if ensembl_count > 0:
                print(f"\n  ⚠ Found {ensembl_count} genes starting with 'ENS' (likely Ensembl IDs)")
                ensembl_genes = gene_names[gene_names.str.startswith('ENS')]
                print(f"     Example: {ensembl_genes[0] if len(ensembl_genes) > 0 else 'N/A'}")
            
            # Check for MT genes with different patterns
            mt_patterns = ['MT-', 'mt-', 'Mt-', 'MT_', 'mt_', 'MT.', 'mt.', 'chrM']
            mt_found = False
            for pattern in mt_patterns:
                matches = gene_names.str.startswith(pattern)
                if matches.any():
                    mt_found = True
                    print(f"\n  ✓ Found {matches.sum()} genes starting with '{pattern}'")
                    print(f"     Examples: {gene_names[matches].tolist()[:5]}")
                    break
            
            if not mt_found:
                # Check if MT genes might be named differently
                contains_mt = gene_names.str.contains('MT|mt|Mitochondr|MITOCHONDR|chrM', 
                                                      case=False, regex=True, na=False)
                if contains_mt.any():
                    print(f"\n  ⚠ Found {contains_mt.sum()} genes containing 'MT' or 'mitochondr'")
                    print(f"     Examples: {gene_names[contains_mt].tolist()[:10]}")
                else:
                    print(f"\n  ⚠ No MT genes found with common patterns")
            
            # Check for ribosomal genes
            ribo_patterns = ['RPS', 'RPL', 'rps', 'rpl', 'Rps', 'Rpl']
            ribo_found = False
            for pattern in ribo_patterns:
                matches = gene_names.str.startswith(pattern)
                if matches.any():
                    ribo_found = True
                    print(f"\n  ✓ Found {matches.sum()} genes starting with '{pattern}'")
                    print(f"     Examples: {gene_names[matches].tolist()[:5]}")
                    break
            
            if not ribo_found:
                contains_ribo = gene_names.str.contains('ribosom|Ribosom|RIBOSOM', 
                                                        case=False, regex=True, na=False)
                if contains_ribo.any():
                    print(f"\n  ⚠ Found {contains_ribo.sum()} genes containing 'ribosom'")
                    print(f"     Examples: {gene_names[contains_ribo].tolist()[:10]}")
                else:
                    print(f"\n  ⚠ No ribosomal genes found with common patterns")
            
            # Check var metadata columns
            print(f"\n  Available var (gene) metadata columns:")
            for col in adata.var.columns:
                print(f"    - {col}")
            
            # Detailed inspection of feature columns
            feature_cols = ['feature_name', 'feature_biotype', 'feature_type']
            found_feature_cols = [col for col in feature_cols if col in adata.var.columns]
            
            if found_feature_cols:
                print(f"\n  🔍 Detailed inspection of feature columns:")
                print(f"  {'='*80}")
                
                # Show top 10 rows for each feature column
                for col in found_feature_cols:
                    print(f"\n  📋 Column: {col}")
                    print(f"     Top 10 values:")
                    values = adata.var[col].head(10)
                    for i, val in enumerate(values, 1):
                        print(f"       {i:2d}. {val}")
                    
                    # Check if this column contains gene symbols (not Ensembl IDs)
                    if col in ['feature_name', 'feature_reference']:
                        # Check if values look like gene symbols vs Ensembl IDs
                        sample_vals = adata.var[col].head(100)
                        ensembl_in_col = sample_vals.astype(str).str.startswith('ENS').sum()
                        symbol_like = sample_vals.astype(str).str.match(r'^[A-Z][A-Z0-9-]+$').sum()
                        
                        if ensembl_in_col < len(sample_vals) * 0.5:  # Less than 50% are Ensembl
                            print(f"     → This column may contain gene symbols (not Ensembl IDs)")
                            # Check for MT genes in this column
                            mt_in_col = adata.var[col].astype(str).str.startswith('MT-', na=False).sum()
                            if mt_in_col > 0:
                                print(f"     → Found {mt_in_col} genes starting with 'MT-' in this column")
                                mt_examples = adata.var[adata.var[col].astype(str).str.startswith('MT-', na=False)][col].head(5)
                                print(f"       Examples: {mt_examples.tolist()}")
                
                # Check biotype/type columns for mitochondrial and ribosomal annotations
                biotype_cols = [col for col in found_feature_cols if 'biotype' in col.lower() or 'type' in col.lower()]
                if biotype_cols:
                    print(f"\n  🧬 Checking biotype/type columns for MT/Ribo annotations:")
                    for col in biotype_cols:
                        unique_biotypes = adata.var[col].unique()
                        print(f"\n     Column: {col}")
                        print(f"     Unique values (showing first 20): {list(unique_biotypes[:20])}")
                        
                        # Check for mitochondrial-related biotypes
                        mt_biotypes = adata.var[col].astype(str).str.contains(
                            'mitochondr|MT|mt', case=False, regex=True, na=False
                        )
                        if mt_biotypes.any():
                            print(f"     → Found {mt_biotypes.sum()} genes with mitochondrial-related biotype")
                            mt_bio_examples = adata.var[mt_biotypes][col].unique()[:5]
                            print(f"       Examples: {list(mt_bio_examples)}")
                        
                        # Check for ribosomal-related biotypes
                        ribo_biotypes = adata.var[col].astype(str).str.contains(
                            'ribosom|ribo|RPS|RPL', case=False, regex=True, na=False
                        )
                        if ribo_biotypes.any():
                            print(f"     → Found {ribo_biotypes.sum()} genes with ribosomal-related biotype")
                            ribo_bio_examples = adata.var[ribo_biotypes][col].unique()[:5]
                            print(f"       Examples: {list(ribo_bio_examples)}")
             
            adata.file.close()  # Close the backed file
            
        except Exception as e:
            print(f"  ✗ Error reading {file_name}: {str(e)}")
            import traceback
            traceback.print_exc()
    
    print("\n" + "=" * 80)

# Run the diagnostic
check_gene_names_in_files(INPUT_DIR)

In [ ]:
def preprocess_h5ad_files(input_dir, output_dir, 
                          female_keywords=['female', 'F', 'f', 'Female', 'FEMALE'],
                          mt_threshold=0.20, 
                          ribo_threshold=0.20):
    """
    Preprocess h5ad files: filter female donors, remove doublets, filter by MT/ribo content,
    filter cells with <400 genes, and filter genes expressed in <3 cells.
    
    Parameters:
    -----------
    input_dir : str
        Directory containing input h5ad files
    output_dir : str
        Directory to save processed h5ad files
    female_keywords : list
        Keywords to identify female donors in metadata
    mt_threshold : float
        Maximum allowed mitochondrial RNA percentage (default: 0.20 = 20%)
    ribo_threshold : float
        Maximum allowed ribosomal RNA percentage (default: 0.20 = 20%)
    """
    import glob
    from scipy.sparse import issparse
    
    # Get all h5ad files
    h5ad_files = glob.glob(os.path.join(input_dir, "*.h5ad"))
    
    if not h5ad_files:
        print(f"No .h5ad files found in {input_dir}")
        return
    
    print(f"Found {len(h5ad_files)} .h5ad file(s)\n")
    
    # Set scanpy settings
    sc.settings.verbosity = 3  # verbosity level
    sc.settings.set_figure_params(dpi=80, facecolor='white')
    
    for h5ad_file in h5ad_files:
        file_name = os.path.basename(h5ad_file)
        print(f"=" * 60)
        print(f"Processing: {file_name}")
        print(f"=" * 60)
        
        try:
            # Read h5ad file
            adata = sc.read_h5ad(h5ad_file)
            print(f"  Initial cells: {adata.n_obs}, genes: {adata.n_vars}")
            
            # Step 1: Filter for female donors
            print("\n1. Filtering for female donors...")
            
            # Common column names for sex/gender
            sex_columns = ['sex', 'gender', 'donor_sex', 'Sex', 'Gender', 
                          'Donor_sex', 'donor_sex', 'sample_sex', 'Sample_sex']
            
            female_mask = None
            sex_col = None
            
            for col in sex_columns:
                if col in adata.obs.columns:
                    sex_col = col
                    # Create boolean mask for female
                    female_mask = adata.obs[col].astype(str).str.lower().isin(
                        [kw.lower() for kw in female_keywords]
                    )
                    print(f"  Found sex column: '{col}'")
                    print(f"  Female samples: {female_mask.sum()}/{len(female_mask)}")
                    break
            
            if female_mask is None:
                print("  ⚠ WARNING: No sex/gender column found. Skipping female filter.")
                print("  Available columns:", list(adata.obs.columns))
            else:
                # Only filter if not all cells are female
                n_female = female_mask.sum()
                n_total = len(female_mask)
                
                if n_female < n_total:
                    # Some cells need to be filtered out
                    adata = adata[female_mask].copy()
                    print(f"  After female filter: {adata.n_obs} cells")
                else:
                    # All cells are female - no filtering needed (avoids memory error)
                    print("  All cells are female - skipping filter step")
            
            if adata.n_obs == 0:
                print("  ✗ No cells remaining after filtering. Skipping file.")
                continue
            
            # Step 2: Calculate mitochondrial and ribosomal percentages
            print("\n2. Calculating mitochondrial and ribosomal RNA percentages...")
            
            # Use feature_name column for gene symbols (contains gene symbols, not Ensembl IDs)
            if 'feature_name' not in adata.var.columns:
                print("  ⚠ WARNING: 'feature_name' column not found. Using var_names instead.")
                gene_symbols = adata.var_names.astype(str)
            else:
                gene_symbols = adata.var['feature_name'].astype(str)
                print(f"  Using 'feature_name' column for gene symbol detection")
            
            # Calculate mitochondrial genes (case-insensitive) using gene symbols
            adata.var['mt'] = gene_symbols.str.startswith('MT-') | \
                             gene_symbols.str.startswith('mt-') | \
                             gene_symbols.str.startswith('Mt-')
            
            # Calculate ribosomal genes (case-insensitive) using gene symbols
            adata.var['ribo'] = gene_symbols.str.startswith('RPS') | \
                               gene_symbols.str.startswith('RPL') | \
                               gene_symbols.str.startswith('rps') | \
                               gene_symbols.str.startswith('rpl') | \
                               gene_symbols.str.startswith('Rps') | \
                               gene_symbols.str.startswith('Rpl')
            
            # Report how many MT and Ribo genes were found
            n_mt = adata.var['mt'].sum()
            n_ribo = adata.var['ribo'].sum()
            print(f"  Found {n_mt} mitochondrial genes and {n_ribo} ribosomal genes")
            
            # Calculate percentages
            sc.pp.calculate_qc_metrics(adata, percent_top=None, log1p=False, inplace=True)
            sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
            sc.pp.calculate_qc_metrics(adata, qc_vars=['ribo'], percent_top=None, log1p=False, inplace=True)
            
            # Get percentage columns (scanpy adds these)
            mt_pct_col = None
            ribo_pct_col = None
            
            for col in adata.obs.columns:
                if 'pct_counts_mt' in col.lower() or 'percent_mito' in col.lower():
                    mt_pct_col = col
                if 'pct_counts_ribo' in col.lower() or 'percent_ribo' in col.lower():
                    ribo_pct_col = col
            
            # If scanpy didn't create them, calculate manually
            if mt_pct_col is None:
                if issparse(adata.X):
                    mt_counts = adata[:, adata.var['mt']].X.sum(axis=1).A1
                    total_counts = adata.X.sum(axis=1).A1
                else:
                    mt_counts = adata[:, adata.var['mt']].X.sum(axis=1)
                    total_counts = adata.X.sum(axis=1)
                adata.obs['pct_counts_mt'] = (mt_counts / total_counts) * 100
            
            if ribo_pct_col is None:
                if issparse(adata.X):
                    ribo_counts = adata[:, adata.var['ribo']].X.sum(axis=1).A1
                    total_counts = adata.X.sum(axis=1).A1
                else:
                    ribo_counts = adata[:, adata.var['ribo']].X.sum(axis=1)
                    total_counts = adata.X.sum(axis=1)
                adata.obs['pct_counts_ribo'] = (ribo_counts / total_counts) * 100
            
            # Use the column names we found or created
            mt_col = mt_pct_col if mt_pct_col else 'pct_counts_mt'
            ribo_col = ribo_pct_col if ribo_pct_col else 'pct_counts_ribo'
            
            # Convert to percentage if needed (scanpy sometimes uses 0-1 scale)
            if adata.obs[mt_col].max() <= 1:
                adata.obs[mt_col] = adata.obs[mt_col] * 100
            if adata.obs[ribo_col].max() <= 1:
                adata.obs[ribo_col] = adata.obs[ribo_col] * 100
            
            print(f"  MT% range: {adata.obs[mt_col].min():.2f} - {adata.obs[mt_col].max():.2f}%")
            print(f"  Ribo% range: {adata.obs[ribo_col].min():.2f} - {adata.obs[ribo_col].max():.2f}%")
            
            # Step 3: Remove doublets using scrublet
            print("\n3. Detecting doublets with scrublet...")
            
            try:
                # Prepare data for scrublet (need raw counts)
                if 'counts' in adata.layers:
                    counts_matrix = adata.layers['counts']
                elif 'raw' in adata.uns:
                    # Try to get raw data
                    counts_matrix = adata.raw.X if hasattr(adata, 'raw') and adata.raw is not None else adata.X
                else:
                    counts_matrix = adata.X
                
                # Convert to dense if sparse (scrublet needs dense)
                if issparse(counts_matrix):
                    counts_matrix = counts_matrix.toarray()
                
                # Run scrublet
                scrub = scr.Scrublet(counts_matrix, expected_doublet_rate=0.06)
                doublet_scores, predicted_doublets = scrub.scrub_doublets(min_counts=2, 
                                                                          min_cells=3,
                                                                          min_gene_variability_pctl=85,
                                                                          n_prin_comps=30)
                
                # Store results
                adata.obs['doublet_score'] = doublet_scores
                adata.obs['predicted_doublet'] = predicted_doublets
                
                n_doublets = predicted_doublets.sum()
                print(f"  Detected {n_doublets} doublets ({n_doublets/len(predicted_doublets)*100:.2f}%)")
                
                # Filter out doublets - only if there are any
                if n_doublets > 0:
                    # Use indexing without explicit copy - more memory efficient
                    keep_cells = ~predicted_doublets
                    adata = adata[keep_cells]
                    # Force materialization by accessing shape (avoids memory bug)
                    _ = adata.X.shape
                    print(f"  After doublet removal: {adata.n_obs} cells")
                else:
                    print("  No doublets detected - skipping filter step")
                
            except Exception as e:
                print(f"  ⚠ WARNING: Scrublet failed: {str(e)}")
                print("  Continuing without doublet removal...")
            
            if adata.n_obs == 0:
                print("  ✗ No cells remaining after doublet removal. Skipping file.")
                continue
            
            # Step 4: Filter by mitochondrial and ribosomal RNA
            print("\n4. Filtering cells with >20% MT or Ribo RNA...")
            
            # Filter: keep cells with <= threshold
            mt_filter = adata.obs[mt_col] <= (mt_threshold * 100)
            ribo_filter = adata.obs[ribo_col] <= (ribo_threshold * 100)
            combined_filter = mt_filter & ribo_filter
            
            n_filtered = (~combined_filter).sum()
            print(f"  Cells to remove: {n_filtered} (MT>{mt_threshold*100}% or Ribo>{ribo_threshold*100}%)")
            
            # Only filter if some cells need to be removed
            if n_filtered > 0:
                # Use indexing without explicit copy - more memory efficient
                adata = adata[combined_filter]
                # Force materialization by accessing shape (avoids memory bug)
                _ = adata.X.shape
                print(f"  After MT/Ribo filter: {adata.n_obs} cells")
            else:
                print("  All cells pass MT/Ribo filter - skipping filter step")
            
            if adata.n_obs == 0:
                print("  ✗ No cells remaining after filtering. Skipping file.")
                continue
            
            # Step 5: Filter cells with less than 400 genes
            print("\n5. Filtering cells with <400 genes...")
            
            # Calculate number of genes expressed per cell
            if issparse(adata.X):
                n_genes_per_cell = (adata.X > 0).sum(axis=1).A1
            else:
                n_genes_per_cell = (adata.X > 0).sum(axis=1)
            
            adata.obs['n_genes'] = n_genes_per_cell
            
            # Filter cells with at least 400 genes
            min_genes = 400
            cells_before = adata.n_obs
            cells_to_remove = (adata.obs['n_genes'] < min_genes).sum()
            
            if cells_to_remove > 0:
                # Use indexing without explicit copy - more memory efficient
                gene_filter_mask = adata.obs['n_genes'] >= min_genes
                adata = adata[gene_filter_mask]
                # Force materialization by accessing shape (avoids memory bug)
                _ = adata.X.shape
                print(f"  Removed {cells_to_remove} cells with <{min_genes} genes")
                print(f"  After gene filter: {adata.n_obs} cells")
            else:
                print(f"  All cells have >= {min_genes} genes - skipping filter step")
            
            if adata.n_obs == 0:
                print("  ✗ No cells remaining after filtering. Skipping file.")
                continue
            
            # Step 6: Filter genes expressed in fewer than 3 cells
            print("\n6. Filtering genes expressed in <3 cells...")
            
            # Calculate number of cells expressing each gene
            if issparse(adata.X):
                n_cells_per_gene = (adata.X > 0).sum(axis=0).A1
            else:
                n_cells_per_gene = (adata.X > 0).sum(axis=0)
            
            adata.var['n_cells'] = n_cells_per_gene
            
            # Filter genes expressed in at least 3 cells
            min_cells = 3
            genes_before = adata.n_vars
            genes_to_remove = (adata.var['n_cells'] < min_cells).sum()
            
            if genes_to_remove > 0:
                # Use indexing without explicit copy - more memory efficient
                gene_expr_filter_mask = adata.var['n_cells'] >= min_cells
                adata = adata[:, gene_expr_filter_mask]
                # Force materialization by accessing shape (avoids memory bug)
                _ = adata.X.shape
                print(f"  Removed {genes_to_remove} genes expressed in <{min_cells} cells")
                print(f"  After gene expression filter: {adata.n_vars} genes")
            else:
                print(f"  All genes expressed in >= {min_cells} cells - skipping filter step")
            
            # Step 7: Save processed file
            print("\n7. Saving processed file...")
            output_file = os.path.join(output_dir, file_name)
            adata.write(output_file)
            print(f"  ✓ Saved: {output_file}")
            print(f"  Final: {adata.n_obs} cells, {adata.n_vars} genes\n")
            
        except Exception as e:
            print(f"  ✗ Error processing {file_name}: {str(e)}")
            import traceback
            traceback.print_exc()
            continue
    
    print("=" * 60)
    print("✓ Preprocessing complete!")
    print("=" * 60)

# Run preprocessing
preprocess_h5ad_files(INPUT_DIR, OUTPUT_DIR)

In [ ]:
# Single-file preprocessing function (for debugging/reprocessing individual files)

def preprocess_single_h5ad_file(input_dir, output_dir, filename,
                                female_keywords=['female', 'F', 'f', 'Female', 'FEMALE'],
                                mt_threshold=0.20, 
                                ribo_threshold=0.20,
                                gene_symbol_source=None):
    """
    Preprocess a single h5ad file with memory-efficient fixes.

    Parameters:
    -----------
    input_dir : str
        Directory containing input h5ad file
    output_dir : str
        Directory to save processed h5ad file
    filename : str
        Name of the h5ad file to process
    female_keywords : list
        Keywords to identify female donors in metadata
    mt_threshold : float
        Maximum allowed mitochondrial RNA percentage (default: 0.20 = 20%)
    ribo_threshold : float
        Maximum allowed ribosomal RNA percentage (default: 0.20 = 20%)
    gene_symbol_source : str or None
        Source for gene symbols for MT/ribo detection. Use 'var_names' when
        adata.var_names are gene symbols (e.g. ovary_1.h5ad). Use None to
        auto-detect (feature_name column if present, else var_names).
    """
    from scipy.sparse import issparse
    
    h5ad_file = os.path.join(input_dir, filename)
    
    if not os.path.exists(h5ad_file):
        print(f"File not found: {h5ad_file}")
        return
    
    print(f"=" * 60)
    print(f"Processing: {filename}")
    print(f"=" * 60)
    
    try:
        # Read h5ad file
        adata = sc.read_h5ad(h5ad_file)
        print(f"  Initial cells: {adata.n_obs}, genes: {adata.n_vars}")
        
        # Step 1: Filter for female donors
        print("\n1. Filtering for female donors...")
        
        # Common column names for sex/gender
        sex_columns = ['sex', 'gender', 'donor_sex', 'Sex', 'Gender', 
                      'Donor_sex', 'donor_sex', 'sample_sex', 'Sample_sex']
        
        female_mask = None
        sex_col = None
        
        for col in sex_columns:
            if col in adata.obs.columns:
                sex_col = col
                # Create boolean mask for female
                female_mask = adata.obs[col].astype(str).str.lower().isin(
                    [kw.lower() for kw in female_keywords]
                )
                print(f"  Found sex column: '{col}'")
                print(f"  Female samples: {female_mask.sum()}/{len(female_mask)}")
                break
        
        if female_mask is None:
            print("  ⚠ WARNING: No sex/gender column found. Skipping female filter.")
            print("  Available columns:", list(adata.obs.columns))
        else:
            # Only filter if not all cells are female
            n_female = female_mask.sum()
            n_total = len(female_mask)
            
            if n_female < n_total:
                # Some cells need to be filtered out
                adata = adata[female_mask]
                # Materialize view before continuing
                if hasattr(adata, 'is_view') and adata.is_view:
                    _ = adata.X.shape  # Force materialization
                print(f"  After female filter: {adata.n_obs} cells")
            else:
                print("  All cells are female - skipping filter step")
        
        if adata.n_obs == 0:
            print("  ✗ No cells remaining after filtering. Skipping file.")
            return
        
        # Step 2: Calculate mitochondrial and ribosomal percentages
        print("\n2. Calculating mitochondrial and ribosomal RNA percentages...")
        
        # Ensure we're not working with a view before modifying var
        if hasattr(adata, 'is_view') and adata.is_view:
            _ = adata.X.shape  # Force materialization
        
        # Gene symbols for MT/ribo detection (var_names or feature_name column)
        if gene_symbol_source == 'var_names':
            gene_symbols = adata.var_names.astype(str)
            print("  Using var_names for gene symbol detection")
        elif gene_symbol_source is None and 'feature_name' in adata.var.columns:
            gene_symbols = adata.var['feature_name'].astype(str)
            print("  Using 'feature_name' column for gene symbol detection")
        else:
            if gene_symbol_source is None:
                print("  ⚠ WARNING: 'feature_name' column not found. Using var_names instead.")
            gene_symbols = adata.var_names.astype(str)
        
        # Calculate mitochondrial genes (case-insensitive) using gene symbols
        adata.var['mt'] = gene_symbols.str.startswith('MT-') | \
                         gene_symbols.str.startswith('mt-') | \
                         gene_symbols.str.startswith('Mt-')
        
        # Calculate ribosomal genes (case-insensitive) using gene symbols
        adata.var['ribo'] = gene_symbols.str.startswith('RPS') | \
                           gene_symbols.str.startswith('RPL') | \
                           gene_symbols.str.startswith('rps') | \
                           gene_symbols.str.startswith('rpl') | \
                           gene_symbols.str.startswith('Rps') | \
                           gene_symbols.str.startswith('Rpl')
        
        # Report how many MT and Ribo genes were found
        n_mt = adata.var['mt'].sum()
        n_ribo = adata.var['ribo'].sum()
        print(f"  Found {n_mt} mitochondrial genes and {n_ribo} ribosomal genes")
        
        # Calculate percentages
        sc.pp.calculate_qc_metrics(adata, percent_top=None, log1p=False, inplace=True)
        sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
        sc.pp.calculate_qc_metrics(adata, qc_vars=['ribo'], percent_top=None, log1p=False, inplace=True)
        
        # Get percentage columns (scanpy adds these)
        mt_pct_col = None
        ribo_pct_col = None
        
        for col in adata.obs.columns:
            if 'pct_counts_mt' in col.lower() or 'percent_mito' in col.lower():
                mt_pct_col = col
            if 'pct_counts_ribo' in col.lower() or 'percent_ribo' in col.lower():
                ribo_pct_col = col
        
        # If scanpy didn't create them, calculate manually
        if mt_pct_col is None:
            if issparse(adata.X):
                mt_counts = adata[:, adata.var['mt']].X.sum(axis=1).A1
                total_counts = adata.X.sum(axis=1).A1
            else:
                mt_counts = adata[:, adata.var['mt']].X.sum(axis=1)
                total_counts = adata.X.sum(axis=1)
            adata.obs['pct_counts_mt'] = (mt_counts / total_counts) * 100
        
        if ribo_pct_col is None:
            if issparse(adata.X):
                ribo_counts = adata[:, adata.var['ribo']].X.sum(axis=1).A1
                total_counts = adata.X.sum(axis=1).A1
            else:
                ribo_counts = adata[:, adata.var['ribo']].X.sum(axis=1)
                total_counts = adata.X.sum(axis=1)
            adata.obs['pct_counts_ribo'] = (ribo_counts / total_counts) * 100
        
        # Use the column names we found or created
        mt_col = mt_pct_col if mt_pct_col else 'pct_counts_mt'
        ribo_col = ribo_pct_col if ribo_pct_col else 'pct_counts_ribo'
        
        # Convert to percentage if needed (scanpy sometimes uses 0-1 scale)
        if adata.obs[mt_col].max() <= 1:
            adata.obs[mt_col] = adata.obs[mt_col] * 100
        if adata.obs[ribo_col].max() <= 1:
            adata.obs[ribo_col] = adata.obs[ribo_col] * 100
        
        print(f"  MT% range: {adata.obs[mt_col].min():.2f} - {adata.obs[mt_col].max():.2f}%")
        print(f"  Ribo% range: {adata.obs[ribo_col].min():.2f} - {adata.obs[ribo_col].max():.2f}%")
        
        # Step 3: Remove doublets using scrublet
        print("\n3. Detecting doublets with scrublet...")
        
        try:
            # Prepare data for scrublet (need raw counts)
            if 'counts' in adata.layers:
                counts_matrix = adata.layers['counts']
            elif 'raw' in adata.uns:
                # Try to get raw data
                counts_matrix = adata.raw.X if hasattr(adata, 'raw') and adata.raw is not None else adata.X
            else:
                counts_matrix = adata.X
            
            # Convert to dense if sparse (scrublet needs dense)
            if issparse(counts_matrix):
                counts_matrix = counts_matrix.toarray()
            
            # Run scrublet
            scrub = scr.Scrublet(counts_matrix, expected_doublet_rate=0.06)
            doublet_scores, predicted_doublets = scrub.scrub_doublets(min_counts=2, 
                                                                      min_cells=3,
                                                                      n_prin_comps=30)
            
            # Store results
            adata.obs['doublet_score'] = doublet_scores
            adata.obs['predicted_doublet'] = predicted_doublets
            
            n_doublets = predicted_doublets.sum()
            print(f"  Detected {n_doublets} doublets ({n_doublets/len(predicted_doublets)*100:.2f}%)")
            
            # Filter out doublets - only if there are any
            if n_doublets > 0:
                # Use indexing without explicit copy - more memory efficient
                keep_cells = ~predicted_doublets
                adata = adata[keep_cells]
                # Force materialization by accessing shape (avoids memory bug)
                if hasattr(adata, 'is_view') and adata.is_view:
                    _ = adata.X.shape
                print(f"  After doublet removal: {adata.n_obs} cells")
            else:
                print("  No doublets detected - skipping filter step")
            
        except Exception as e:
            print(f"  ⚠ WARNING: Scrublet failed: {str(e)}")
            print("  Continuing without doublet removal...")
        
        if adata.n_obs == 0:
            print("  ✗ No cells remaining after doublet removal. Skipping file.")
            return
        
        # Step 4: Filter by mitochondrial and ribosomal RNA
        print("\n4. Filtering cells with >20% MT or Ribo RNA...")
        
        # Ensure we're not working with a view
        if hasattr(adata, 'is_view') and adata.is_view:
            _ = adata.X.shape
        
        # Filter: keep cells with <= threshold
        mt_filter = adata.obs[mt_col] <= (mt_threshold * 100)
        ribo_filter = adata.obs[ribo_col] <= (ribo_threshold * 100)
        combined_filter = mt_filter & ribo_filter
        
        n_filtered = (~combined_filter).sum()
        print(f"  Cells to remove: {n_filtered} (MT>{mt_threshold*100}% or Ribo>{ribo_threshold*100}%)")
        
        # Only filter if some cells need to be removed
        if n_filtered > 0:
            # Use indexing without explicit copy - more memory efficient
            adata = adata[combined_filter]
            # Force materialization by accessing shape (avoids memory bug)
            if hasattr(adata, 'is_view') and adata.is_view:
                _ = adata.X.shape
            print(f"  After MT/Ribo filter: {adata.n_obs} cells")
        else:
            print("  All cells pass MT/Ribo filter - skipping filter step")
        
        if adata.n_obs == 0:
            print("  ✗ No cells remaining after filtering. Skipping file.")
            return
        
        # Step 5: Filter cells with fewer than 400 genes
        print("\n5. Filtering cells with <400 genes...")
        if hasattr(adata, 'is_view') and adata.is_view:
            _ = adata.X.shape
        if issparse(adata.X):
            n_genes_per_cell = (adata.X > 0).sum(axis=1).A1
        else:
            n_genes_per_cell = (adata.X > 0).sum(axis=1)
        min_genes = 400
        gene_filter_mask = n_genes_per_cell >= min_genes
        cells_to_remove = (~gene_filter_mask).sum()
        if cells_to_remove > 0:
            adata = adata[gene_filter_mask]
            _ = adata.X.shape
            adata.obs['n_genes'] = n_genes_per_cell[gene_filter_mask]
            print(f"  Removed {cells_to_remove} cells with <{min_genes} genes")
            print(f"  After gene filter: {adata.n_obs} cells")
        else:
            adata.obs['n_genes'] = n_genes_per_cell
            print(f"  All cells have >= {min_genes} genes - skipping filter step")
        if adata.n_obs == 0:
            print("  ✗ No cells remaining after filtering. Skipping file.")
            return
        
        # Step 6: Filter genes expressed in fewer than 3 cells
        print("\n6. Filtering genes expressed in <3 cells...")
        if hasattr(adata, 'is_view') and adata.is_view:
            _ = adata.X.shape
        if issparse(adata.X):
            n_cells_per_gene = (adata.X > 0).sum(axis=0).A1
        else:
            n_cells_per_gene = (adata.X > 0).sum(axis=0)
        adata.var['n_cells'] = n_cells_per_gene
        min_cells = 3
        genes_to_remove = (adata.var['n_cells'] < min_cells).sum()
        if genes_to_remove > 0:
            gene_expr_filter_mask = adata.var['n_cells'] >= min_cells
            adata = adata[:, gene_expr_filter_mask]
            _ = adata.X.shape
            print(f"  Removed {genes_to_remove} genes expressed in <{min_cells} cells")
            print(f"  After gene expression filter: {adata.n_vars} genes")
        else:
            print(f"  All genes expressed in >= {min_cells} cells - skipping filter step")
        
        # Step 7: Save processed file
        print("\n7. Saving processed file...")
        output_file = os.path.join(output_dir, filename)
        adata.write(output_file)
        print(f"  ✓ Saved: {output_file}")
        print(f"  Final: {adata.n_obs} cells, {adata.n_vars} genes\n")
        
    except Exception as e:
        print(f"  ✗ Error processing {filename}: {str(e)}")
        import traceback
        traceback.print_exc()


In [ ]:
# Specify the file to process here
SINGLE_FILE = "breast.h5ad"  # Change this to process a different file

# Run preprocessing on single file
preprocess_single_h5ad_file(INPUT_DIR, OUTPUT_DIR, SINGLE_FILE)

## Single-file preprocessing: ovary_1.h5ad

Run the same preprocessing on **ovary_1.h5ad**, where gene symbols are in `adata.var_names` (no `feature_name` column). Uses `gene_symbol_source='var_names'` so MT/ribo detection uses var names (e.g. MT-ND1, RPS8).

In [ ]:
# Preprocess ovary_1.h5ad (gene symbols in var_names, not feature_name)
SINGLE_FILE_OVARY = "ovary_1.h5ad"
preprocess_single_h5ad_file(
    INPUT_DIR,
    OUTPUT_DIR,
    SINGLE_FILE_OVARY,
    gene_symbol_source="var_names",
)

## Downsampling Function for Large Datasets

This function downsamples large datasets (e.g., breast.h5ad with 2M cells) while:
- Preserving cell-type proportions
- Keeping rare cell states (all cells from rare types)
- Downsampling dense clusters harder than sparse ones

In [ ]:
def downsample_adata_density_aware(adata, 
                                    target_n_cells=None,
                                    target_fraction=None,
                                    cell_type_col='cell_type',
                                    rare_threshold=0.001,
                                    min_cells_per_type=100,
                                    k_neighbors=15,
                                    n_pcs=50,
                                    random_seed=42,
                                    chunk_size=50000):
    """
    Downsample AnnData object using density-aware sampling that preserves rare states
    and downsamples dense clusters more than sparse ones.
    
    MEMORY-EFFICIENT VERSION: Processes data in chunks to avoid memory errors.
    
    Parameters:
    -----------
    adata : AnnData
        Input AnnData object to downsample (can be backed='r' for memory efficiency)
    target_n_cells : int, optional
        Target number of cells after downsampling. If None, uses target_fraction.
    target_fraction : float, optional
        Fraction of cells to keep (0-1). Used if target_n_cells is None.
        Default: 0.2 (keep 20% of cells, suitable for 2M -> 400K)
    cell_type_col : str
        Column name in adata.obs containing cell type annotations
    rare_threshold : float
        Fraction threshold for rare cell types (default: 0.001 = 0.1%)
        All cells from types below this threshold are kept
    min_cells_per_type : int
        Minimum number of cells to keep per cell type (default: 100)
        Ensures rare types are preserved even if they're slightly above threshold
    k_neighbors : int
        Number of neighbors for density calculation (default: 15)
    n_pcs : int
        Number of PCs for dimensionality reduction (default: 50)
    random_seed : int
        Random seed for reproducibility (default: 42)
    chunk_size : int
        Size of chunks for processing common cell types (default: 50000)
        Smaller chunks = less memory, but slower
    
    Returns:
    --------
    AnnData
        Downsampled AnnData object
    """
    import scanpy as sc
    from scipy.sparse import issparse
    import numpy as np
    
    np.random.seed(random_seed)
    
    # Determine target number of cells
    if target_n_cells is None:
        if target_fraction is None:
            target_fraction = 0.2  # Default: keep 20%
        target_n_cells = int(adata.n_obs * target_fraction)
    
    print(f"Downsampling from {adata.n_obs:,} to ~{target_n_cells:,} cells")
    print(f"Target fraction: {target_n_cells/adata.n_obs:.2%}")
    
    # Check if cell_type_col exists
    if cell_type_col not in adata.obs.columns:
        raise ValueError(f"Column '{cell_type_col}' not found in adata.obs")
    
    # Get cell type counts (works with backed mode)
    cell_type_counts = adata.obs[cell_type_col].value_counts()
    total_cells = adata.n_obs
    
    print(f"\nCell type distribution:")
    print(f"  Total cell types: {len(cell_type_counts)}")
    print(f"  Most common: {cell_type_counts.iloc[0]:,} cells ({cell_type_counts.iloc[0]/total_cells:.2%})")
    print(f"  Least common: {cell_type_counts.iloc[-1]:,} cells ({cell_type_counts.iloc[-1]/total_cells:.2%})")
    
    # Identify rare cell types
    rare_mask = (cell_type_counts / total_cells < rare_threshold) | (cell_type_counts < min_cells_per_type)
    rare_types = cell_type_counts[rare_mask].index.tolist()
    common_types = cell_type_counts[~rare_mask].index.tolist()
    
    print(f"\nRare cell types (keeping all cells): {len(rare_types)}")
    if len(rare_types) > 0:
        print(f"  Examples: {rare_types[:5]}")
    print(f"Common cell types (density-based sampling): {len(common_types)}")
    
    # Get indices for rare and common cell types (works with backed mode)
    rare_mask_obs = adata.obs[cell_type_col].isin(rare_types).values
    rare_indices = np.where(rare_mask_obs)[0]
    common_mask_obs = adata.obs[cell_type_col].isin(common_types).values
    common_indices = np.where(common_mask_obs)[0]
    
    n_rare_kept = len(rare_indices)
    target_common = target_n_cells - n_rare_kept
    
    print(f"\nSampling strategy:")
    print(f"  Rare cells to keep: {n_rare_kept:,} (all)")
    print(f"  Common cells target: {target_common:,}")
    
    if target_common <= 0:
        print("  Warning: Target is smaller than rare cells. Keeping all rare cells only.")
        # Memory-efficient indexing: only load what we need
        return adata[rare_indices]
    
    # For common cell types, process in chunks to avoid memory issues
    print(f"\nComputing density for common cell types (processing in chunks)...")
    
    # Process common cells in chunks
    n_common = len(common_indices)
    n_chunks = (n_common + chunk_size - 1) // chunk_size
    
    print(f"  Processing {n_common:,} common cells in {n_chunks} chunks of ~{chunk_size:,} cells")
    
    all_density_scores = []
    all_chunk_indices = []
    
    # Check if adata is backed - if so, we need to be more careful
    is_backed = hasattr(adata, 'file') and adata.file is not None
    
    for chunk_idx in range(n_chunks):
        start_idx = chunk_idx * chunk_size
        end_idx = min((chunk_idx + 1) * chunk_size, n_common)
        chunk_common_indices = common_indices[start_idx:end_idx]
        
        print(f"  Processing chunk {chunk_idx + 1}/{n_chunks} ({len(chunk_common_indices):,} cells)...")
        
        # Load only this chunk (memory efficient)
        # For backed objects, we need to explicitly create an in-memory copy
        if is_backed:
            # For backed objects, create a new AnnData with explicit data loading
            # This avoids issues with .to_memory() on views
            import anndata as ad
            from scipy.sparse import issparse as scipy_issparse
            
            # Get the chunk indices sorted
            chunk_indices_sorted = np.sort(chunk_common_indices)
            
            # Explicitly load X matrix
            X_chunk = adata[chunk_indices_sorted].X
            # Ensure it's a proper copy, not a view
            if scipy_issparse(X_chunk):
                X_chunk = X_chunk.copy()
            else:
                X_chunk = X_chunk.copy()
            
            # Create new AnnData object with explicit data
            adata_chunk = ad.AnnData(
                X=X_chunk,
                obs=adata.obs.iloc[chunk_indices_sorted].copy(),
                var=adata.var.copy()
            )
            # Copy uns if it exists
            if hasattr(adata, 'uns') and adata.uns:
                adata_chunk.uns = adata.uns.copy()
        else:
            # For in-memory objects, create a view then copy
            adata_chunk_view = adata[chunk_common_indices]
            if hasattr(adata_chunk_view, 'is_view') and adata_chunk_view.is_view:
                adata_chunk = adata_chunk_view.copy()
            else:
                adata_chunk = adata_chunk_view.copy()
        
        # Preprocess for density calculation
        if adata_chunk.n_vars > 5000:
            sc.pp.normalize_total(adata_chunk, target_sum=1e4)
            sc.pp.log1p(adata_chunk)
            sc.pp.highly_variable_genes(adata_chunk, n_top_genes=2000, subset=True)
        else:
            sc.pp.normalize_total(adata_chunk, target_sum=1e4)
            sc.pp.log1p(adata_chunk)
        
        # Compute PCA for this chunk
        sc.tl.pca(adata_chunk, n_comps=n_pcs, use_highly_variable=True)
        
        # Compute k-nearest neighbors (within chunk)
        sc.pp.neighbors(adata_chunk, n_neighbors=min(k_neighbors, len(chunk_common_indices)-1), n_pcs=n_pcs)
        
        # Calculate local density
        distances = adata_chunk.obsp['distances']
        if issparse(distances):
            mean_distances = np.array(distances.mean(axis=1)).flatten()
        else:
            mean_distances = distances.mean(axis=1)
        
        # Convert to density scores
        density_scores = 1 / (mean_distances + 1e-6)
        all_density_scores.append(density_scores)
        all_chunk_indices.append(chunk_common_indices)
        
        # Clean up memory
        del adata_chunk
    
    # Combine density scores from all chunks
    all_density_scores = np.concatenate(all_density_scores)
    
    # Normalize density scores globally
    all_density_scores = (all_density_scores - all_density_scores.min()) / (all_density_scores.max() - all_density_scores.min() + 1e-6)
    
    print(f"  Density range: {all_density_scores.min():.4f} - {all_density_scores.max():.4f}")
    
    # Sample inversely proportional to density
    power = 2.0
    sampling_probs = (1 - all_density_scores) ** power
    sampling_probs = sampling_probs / sampling_probs.sum() * target_common
    sampling_probs = np.clip(sampling_probs, 0, 1)
    
    # Perform weighted random sampling
    print(f"  Performing density-weighted sampling...")
    sampled_mask = np.random.binomial(1, sampling_probs).astype(bool)
    common_sampled_indices = common_indices[sampled_mask]
    
    n_common_sampled = len(common_sampled_indices)
    print(f"  Sampled {n_common_sampled:,} common cells")
    
    # Combine rare and sampled common cells
    final_indices = np.concatenate([rare_indices, common_sampled_indices])
    final_indices = np.sort(final_indices)
    
    print(f"\n✓ Downsampling complete:")
    print(f"  Final cells: {len(final_indices):,}")
    print(f"  Rare cells kept: {n_rare_kept:,}")
    print(f"  Common cells sampled: {n_common_sampled:,}")
    
    # Create downsampled AnnData (only load final indices)
    print("  Loading final downsampled dataset...")
    
    # Load final dataset - handle backed objects explicitly
    if is_backed:
        # For backed objects, explicitly create in-memory copy
        import anndata as ad
        from scipy.sparse import issparse as scipy_issparse
        
        # Get final indices sorted
        final_indices_sorted = np.sort(final_indices)
        
        # Explicitly load X matrix
        X_final = adata[final_indices_sorted].X
        # Ensure it's a proper copy
        if scipy_issparse(X_final):
            X_final = X_final.copy()
        else:
            X_final = X_final.copy()
        
        # Create new AnnData object
        adata_downsampled = ad.AnnData(
            X=X_final,
            obs=adata.obs.iloc[final_indices_sorted].copy(),
            var=adata.var.copy()
        )
        # Copy uns if it exists
        if hasattr(adata, 'uns') and adata.uns:
            adata_downsampled.uns = adata.uns.copy()
        # Preserve .raw attribute if it exists (subset to same cells)
        if adata.raw is not None:
            raw_X = adata.raw.X[final_indices_sorted]
            if scipy_issparse(raw_X):
                raw_X = raw_X.copy()
            else:
                raw_X = np.array(raw_X).copy()
            adata_downsampled.raw = ad.AnnData(X=raw_X, var=adata.raw.var.copy())
    else:
        # For in-memory objects, create view then copy
        adata_downsampled_view = adata[final_indices]
        if hasattr(adata_downsampled_view, 'is_view') and adata_downsampled_view.is_view:
            adata_downsampled = adata_downsampled_view.copy()
        else:
            adata_downsampled = adata_downsampled_view.copy()
    
    # Verify cell type proportions
    print(f"\nCell type proportion preservation:")
    original_props = adata.obs[cell_type_col].value_counts(normalize=True)
    downsampled_props = adata_downsampled.obs[cell_type_col].value_counts(normalize=True)
    
    # Compare proportions for common types
    common_prop_diff = []
    for ct in common_types[:10]:
        orig_p = original_props.get(ct, 0)
        down_p = downsampled_props.get(ct, 0)
        if orig_p > 0:
            diff = abs(orig_p - down_p) / orig_p * 100
            common_prop_diff.append(diff)
            if ct in original_props.index[:5]:
                print(f"  {ct}: {orig_p:.3%} -> {down_p:.3%} (diff: {diff:.1f}%)")
    
    if common_prop_diff:
        print(f"  Mean proportion change for common types: {np.mean(common_prop_diff):.1f}%")
    
    return adata_downsampled

## Example: Downsample breast.h5ad

Load the breast dataset, downsample it, and save the result.

In [ ]:
# Example usage: Downsample breast.h5ad
# This can be integrated into the preprocessing pipeline

BREAST_FILE = "breast.h5ad"
TARGET_CELLS = 400000  # Target ~400K cells (similar to other tissues)

# Load breast data in backed mode for memory efficiency
print("Loading breast.h5ad (backed mode for memory efficiency)...")
breast_path = str(pp.PREPROCESSED_H5AD / "breast.h5ad")  # optional example; set your own path
if os.path.exists(breast_path):
    # Load in backed mode to avoid loading full dataset into memory
    breast_adata = sc.read_h5ad(breast_path, backed='r')
    print(f"Loaded: {breast_adata.n_obs:,} cells, {breast_adata.n_vars:,} genes")
    
    # Check if cell_type column exists
    cell_type_col = None
    for col in ['cell_type', 'author_cell_type', 'celltype', 'CellType']:
        if col in breast_adata.obs.columns:
            cell_type_col = col
            break
    
    if cell_type_col is None:
        print("⚠️ Warning: No cell type column found. Using uniform sampling instead.")
        # Fallback: simple uniform sampling
        if breast_adata.n_obs > TARGET_CELLS:
            import numpy as np
            np.random.seed(42)
            indices = np.random.choice(breast_adata.n_obs, TARGET_CELLS, replace=False)
            # Load only selected indices
            breast_downsampled = sc.read_h5ad(breast_path)[indices]
        else:
            breast_downsampled = sc.read_h5ad(breast_path)
    else:
        print(f"Using cell type column: {cell_type_col}")
        # Use backed mode - function will handle chunking efficiently
        # The function processes in chunks, so we can pass the backed object
        
        # Downsample using density-aware method (memory-efficient chunking)
        breast_downsampled = downsample_adata_density_aware(
            breast_adata,  # Pass backed object - function handles chunking
            target_n_cells=TARGET_CELLS,
            cell_type_col=cell_type_col,
            rare_threshold=0.001,  # Keep all types < 0.1% of total
            min_cells_per_type=100,  # Keep at least 100 cells per type
            k_neighbors=15,
            n_pcs=50,
            random_seed=42,
            chunk_size=50000  # Process 50K cells at a time (adjust if memory issues persist)
        )
    
    # Save downsampled data
    output_path = os.path.join(INPUT_DIR, BREAST_FILE)
    print(f"\nSaving downsampled data to: {output_path}")
    breast_downsampled.write(output_path)
    print(f"✓ Saved: {breast_downsampled.n_obs:,} cells, {breast_downsampled.n_vars:,} genes")
    
    # Close backed file if it was opened
    if hasattr(breast_adata, 'file'):
        breast_adata.file.close()
else:
    print(f"File not found: {breast_path}")